# L1: Servo-driven loaded pendulum

This notebook inspects one deterministic run cell by cell. The declared boundary is `q_des -> fixed PD torque command -> command delay -> pendulum -> encoder q`, with `qd` derived from encoder position. Fit observations are public; Oracle torque and delayed-command state remain diagnostics.


In [ ]:
from pathlib import Path
import sys
from tempfile import TemporaryDirectory
from IPython.display import SVG, Image, display
repo = Path.cwd() if (Path.cwd() / 'synthetic').is_dir() else Path.cwd().parent
sys.path.insert(0, str(repo))
from synthetic.l1_servo_loaded_pendulum import (
    load_config, run_l1, run_summary, machine_frame, machine_svg, plot_run,
)
config = load_config()
run = run_l1(config)
summary = run_summary(run)
summary['validation']


## Machine/state agreement

The machine frame and plots read the same recorded trajectory. Change `index` to inspect another replay state; this does not simulate or fit.


In [ ]:
index = len(run.data.validation.t) // 2
frame = machine_frame(run, split='validation', index=index, role='oracle')
frame


In [ ]:
svg = machine_svg(run, split='validation', index=index)
display(SVG(svg))


## Recorded report

`plot_run` consumes the completed `Run` artifact and writes synchronized fit, validation, torque-diagnostic, residual, and machine views.


In [ ]:
with TemporaryDirectory(prefix='l1-notebook-') as output:
    report_path = plot_run(run, Path(output) / 'report.png')
    display(Image(filename=str(report_path)))


## Separate public fitting path

The next cell deliberately constructs the estimator call from public fit observations and `Config.model`. It does not pass `run.config.truth`, Oracle trajectories, validation observations, torque, or hidden command state.


In [ ]:
from synthetic.l1_servo_loaded_pendulum import fit_student
public_fit = run.data.fit
student_fit = fit_student(
    public_fit,
    config.model,
    optimizer_start=config.optimizer_start,
    lower=config.lower,
    upper=config.upper,
)
student_fit


In [ ]:
{
    'oracle_delay_for_evaluation': run.config.truth.delay_s,
    'identified_delay_from_public_fit': student_fit.params.delay_s,
    'fit_q_rmse_initial': run.initial_fit.q_rmse,
    'fit_q_rmse_identified': run.identified_fit.q_rmse,
    'validation_q_rmse_initial': run.initial_validation.q_rmse,
    'validation_q_rmse_identified': run.identified_validation.q_rmse,
}


## Exercise

Change the learner-visible Initial delay in the Marimo app, predict how the orange baseline and residual phase will move, then press **Run identification**. A presentation or timeline change replays the completed run; only the explicit fit action creates a new identified result.
